# 3D RM-synthesis

In [ ]:
from __future__ import annotations

import tempfile
from pathlib import Path

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import zarr
from astropy.io import fits
from astropy.visualization import quantity_support

plt.rcParams["figure.dpi"] = 150

_ = quantity_support()
rng = np.random.default_rng(42)

We'll simulate a small Stokes Q/U cutout cube with a per-pixel Faraday rotation measure, write it to FITS with a dummy Stokes axis -- as in a typical ASKAP/EMU cutout cube (e.g. `image.restored.q.<field>.fits`) -- and read it back lazily with `rm_lite.utils.dask_io`.

As with the 1D example, we'll simulate RACS-all frequency coverage.

In [ ]:
bw_low = 288
freqs = np.linspace(943.5 - bw_low / 2, 943.5 + bw_low / 2, 36) * u.MHz
freq_hz = freqs.to(u.Hz).value

Now build a small image with two compact polarised sources -- one bottom-right, one top-left -- each a 2D Gaussian blob, using `rm_lite.utils.fitting.gaussian` evaluated on a radial distance grid, on an otherwise unpolarised background. Keeping most of the field genuinely source-free matters below: the robust per-channel noise estimator needs mostly-empty sky to lock onto, same as it would on a real image.

In [ ]:
from rm_lite.utils.fitting import gaussian
from rm_lite.utils.synthesis import faraday_simple_spectrum

ny, nx = 64, 64
y_grid, x_grid = np.mgrid[0:ny, 0:nx]
blob_y, blob_x = ny * 0.3, nx * 0.7
blob2_y, blob2_x = ny * 0.7, nx * 0.3
rm_map = (
    80.0 * (x_grid / nx - 0.5) * 2
    + np.exp(-((x_grid - blob_x) ** 2 + (y_grid - blob_y) ** 2) / (2 * 4.0**2))
    - np.exp(-((x_grid - blob2_x) ** 2 + (y_grid - blob2_y) ** 2) / (2 * 4.0**2))
)

radius_grid = np.hypot(x_grid - blob_x, y_grid - blob_y)
radius2_grid = np.hypot(x_grid - blob2_x, y_grid - blob2_y)
frac_pol_map = gaussian(radius_grid, amplitude=0.6, mean=0.0, fwhm=6.0) + gaussian(
    radius2_grid, amplitude=0.6, mean=0.0, fwhm=6.0
)

psi0_deg = 30.0
rms_noise = 0.03

stokes_q = np.empty((freq_hz.size, ny, nx), dtype=np.float32)
stokes_u = np.empty((freq_hz.size, ny, nx), dtype=np.float32)
for j in range(ny):
    for i in range(nx):
        complex_spectrum = faraday_simple_spectrum(
            freq_hz,
            frac_pol=frac_pol_map[j, i],
            psi0_deg=psi0_deg,
            rm_radm2=rm_map[j, i],
        )
        stokes_q[:, j, i] = complex_spectrum.real + rng.normal(
            0, rms_noise, freq_hz.size
        )
        stokes_u[:, j, i] = complex_spectrum.imag + rng.normal(
            0, rms_noise, freq_hz.size
        )

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
im1 = ax1.imshow(rm_map, origin="lower", cmap="RdBu_r", vmin=-100, vmax=100)
fig.colorbar(im1, ax=ax1, label=f"RM / ({u.rad / u.m**2:latex_inline})")
ax1.set(title="Input (true) RM map")
im2 = ax2.imshow(frac_pol_map, origin="lower")
fig.colorbar(im2, ax=ax2, label="Fractional polarisation")
ax2.set(title="Input (true) fractional polarisation")

Now write Stokes Q and U to separate FITS cubes, each with a degenerate leading Stokes axis (`NAXIS4 = 1`).

In [ ]:
tmpdir = Path(tempfile.mkdtemp())


def write_stokes_fits(path: Path, data: np.ndarray, stokes: int) -> None:
    header = fits.Header()
    header["CTYPE1"] = "RA---SIN"
    header["CTYPE2"] = "DEC--SIN"
    header["CTYPE3"] = "FREQ"
    header["CRVAL3"] = freq_hz[0]
    header["CDELT3"] = float(np.diff(freq_hz)[0])
    header["CRPIX3"] = 1.0
    header["CUNIT3"] = "Hz"
    header["CTYPE4"] = "STOKES"
    header["CRVAL4"] = stokes
    header["CDELT4"] = 1.0
    header["CRPIX4"] = 1.0
    fits.PrimaryHDU(data=data[np.newaxis, ...], header=header).writeto(
        path, overwrite=True
    )


stokes_q_fits = tmpdir / "cutout.q.fits"
stokes_u_fits = tmpdir / "cutout.u.fits"
write_stokes_fits(stokes_q_fits, stokes_q, stokes=2)
write_stokes_fits(stokes_u_fits, stokes_u, stokes=3)

fits.getdata(stokes_q_fits).shape

`read_fits_cube_dask` reads the cube one spatial block at a time: each block reopens the FITS file with `memmap=True` and slices out just that block's rows and columns, wrapped in a `dask.delayed` call and assembled into a single chunked `dask.array`. Actual disk reads are deferred until a block is computed. The frequency axis is always kept whole in a chunk -- RM-synthesis needs every channel per pixel -- and the spatial chunk size is picked from a target per-chunk memory footprint, here set artificially small so multiple chunks show up in this small demo cube.

In practice, peak memory during processing is a multiple of `target_chunk_mb` (roughly 2-4x, from buffer copies made when a block is materialised and again when it's written out), not a hard cap at that value -- but it does scale with chunk size, not cube size, so a `target_chunk_mb` set well below your available memory stays safe regardless of how large the on-disk cube is. Total RM-synthesis wall-clock time is set by the cube's total size, not how it's chunked, so smaller chunks cost little in speed -- when in doubt, prefer a smaller `target_chunk_mb`.

In [ ]:
from rm_lite.utils.dask_io import read_fits_cube_dask

q_dask, q_header = read_fits_cube_dask(
    stokes_q_fits, target_chunk_mb=32 * 1024 / 1024**2
)
u_dask, _ = read_fits_cube_dask(stokes_u_fits, target_chunk_mb=32 * 1024 / 1024**2)
q_dask

Before running RM-synthesis, estimate a robust per-channel noise directly from the cube with `estimate_channel_noise_mad` -- it takes the MAD-based standard deviation over every pixel in each channel plane, combined across Q and U the same way `compute_rmsynth_params` combines a per-channel error spectrum. This is what feeds `weight_arr` below and, in the RM-CLEAN notebook, the auto-mask/threshold.

In [ ]:
from rm_lite.utils.dask_io import estimate_channel_noise_mad

channel_noise = estimate_channel_noise_mad(q_dask, u_dask)
weight_arr = 1.0 / channel_noise**2

print(f"true per-channel noise:      {rms_noise:.4f}")
print(f"estimated per-channel noise: {channel_noise.mean():.4f}")

Run 3D RM-synthesis with `rmsynth_3d`. This applies the same `rmsynth_nufft`/`get_rmsf_nufft` math used by the 1D/2D tools across the whole cube via `dask.array.map_blocks`, one call per spatial chunk.

We pin `phi_max_radm2`/`d_phi_radm2` to a small, coarse Faraday depth grid (just enough planes to cover our simulated RM range) so this demo runs quickly -- a real search would use finer sampling.

In [ ]:
from rm_lite.tools_3d.rmsynth import rmsynth_3d

help(rmsynth_3d)

In [ ]:
result = rmsynth_3d(
    q_dask,
    u_dask,
    freq_hz,
    weight_arr=weight_arr,
    phi_max_radm2=150.0,
    d_phi_radm2=10.0,
)
result.fdf_dirty_cube

`rmsynth_3d` takes dask arrays, for callers who already have Q/U loaded (e.g. from zarr). When Q/U are plain FITS files on disk, `rmsynth_3d_from_fits` folds the `read_fits_cube_dask` calls above into `rmsynth_3d` itself, so the two files-to-lazy-cube reads don't need to be written out by hand each time.

In [ ]:
from rm_lite.tools_3d.rmsynth import rmsynth_3d_from_fits

result_from_fits = rmsynth_3d_from_fits(
    stokes_q_fits,
    stokes_u_fits,
    weight_arr=weight_arr,
    phi_max_radm2=150.0,
    d_phi_radm2=10.0,
    target_chunk_mb=32 * 1024 / 1024**2,
)
np.allclose(result_from_fits.fdf_dirty_cube.compute(), result.fdf_dirty_cube.compute())

`fdf_dirty_cube` and `rmsf_cube` are still lazy -- nothing has been computed yet. Let's materialise the dirty FDF cube and look at the peak polarised intensity and recovered RM maps.

In [ ]:
fdf_dirty_cube = result.fdf_dirty_cube.compute()
peak_pi_map = np.max(np.abs(fdf_dirty_cube), axis=0)
peak_idx_map = np.argmax(np.abs(fdf_dirty_cube), axis=0)
peak_rm_map = result.phi_arr_radm2[peak_idx_map]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
im1 = ax1.imshow(peak_pi_map, origin="lower")
fig.colorbar(im1, ax=ax1, label="Polarised intensity")
ax1.set(title="Peak polarised intensity")
im2 = ax2.imshow(peak_rm_map, origin="lower", cmap="RdBu_r", vmin=-100, vmax=100)
fig.colorbar(im2, ax=ax2, label=f"RM / ({u.rad / u.m**2:latex_inline})")
ax2.set(title="Peak RM (recovered)")

## Stokes I fractional-polarisation correction

So far the FDF has been built straight from Stokes Q/U, in flux units. But a source's Stokes I spectral index imprints a *spurious* structure on the FDF: the frequency-dependent total intensity acts as an extra weighting across $\lambda^2$, distorting the FDF main lobe in a way that has nothing to do with real Faraday dispersion. The fix, exactly as in the 1D tools, is to divide by a Stokes I model first -- forming fractional spectra $q = Q/I$, $u = U/I$ -- and then rescale the FDF back to absolute polarised flux by the reference-frequency Stokes I flux.

`rmsynth_3d` does this per pixel when you pass a `stokes_i` cube: it fits a Stokes I model to every pixel with the same fitter the 1D tools use (`rm_lite.utils.fitting.fit_stokes_i_model`), or you can pass a ready-made `stokes_i_model` cube. Here we give the simulated field a spatially varying spectral index, build observed flux-scale Q/U from it, and let `rmsynth_3d` fit and correct it.

In [ ]:
import dask.array as da
from rm_lite.tools_3d.rmsynth import rmsynth_3d

# Give the field a Stokes I with a spatially varying spectral index.
ref_freq_hz = float(np.median(freq_hz))
alpha_map = -1.0 + 0.6 * (x_grid / nx - 0.5) * 2  # ~ -1.6 .. -0.4 across the field
i_ref_map = 0.5 + frac_pol_map  # brighter towards the polarised sources
stokes_i = (
    i_ref_map[None] * (freq_hz / ref_freq_hz)[:, None, None] ** alpha_map[None]
).astype(np.float32)

# Observed flux-scale Q/U = fractional polarisation * Stokes I.
stokes_q_obs = stokes_q * stokes_i
stokes_u_obs = stokes_u * stokes_i
stokes_i_obs = stokes_i + rng.normal(0, rms_noise, stokes_i.shape).astype(np.float32)

chunks = (-1, 16, 16)
result_i = rmsynth_3d(
    da.from_array(stokes_q_obs, chunks=chunks),
    da.from_array(stokes_u_obs, chunks=chunks),
    freq_hz,
    stokes_i=da.from_array(stokes_i_obs, chunks=chunks),
    estimate_stokes_i_noise=True,  # per-channel noise from the I cube
    fit_function="log",  # power-law fit; fit_order=2 by default
    phi_max_radm2=150.0,
    d_phi_radm2=1.0,
    weight_type="uniform",
)
model_cube = result_i.stokes_i_model_cube.compute()
result_i.stokes_i_model_cube, result_i.stokes_i_ref_flux_map

The per-pixel fit runs quietly (the 1D fitter's per-fit logging is suppressed at cube scale) and returns a lazy Stokes I model cube plus the reference-flux map the FDF was rescaled by. Below, the fitted model tracks the observed Stokes I at a bright pixel, and the I-corrected peak polarised-intensity and RM maps recover the input field.

In [ ]:
jb, ib = int(blob_y), int(blob_x)  # a bright polarised pixel
fdf_i = result_i.fdf_dirty_cube.compute()
peak_pi = np.max(np.abs(fdf_i), axis=0)
peak_rm = result_i.phi_arr_radm2[np.argmax(np.abs(fdf_i), axis=0)]

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))
ax1.plot(freq_hz / 1e9, stokes_i_obs[:, jb, ib], ".", ms=4, label="observed I")
ax1.plot(freq_hz / 1e9, model_cube[:, jb, ib], "-", label="fitted model")
ax1.set(xlabel="Frequency / GHz", ylabel="Stokes I", title="Per-pixel Stokes I fit")
ax1.legend()
im2 = ax2.imshow(peak_pi, origin="lower")
fig.colorbar(im2, ax=ax2, label="Polarised intensity")
ax2.set(title="Peak PI (I-corrected)")
im3 = ax3.imshow(peak_rm, origin="lower", cmap="RdBu_r", vmin=-100, vmax=100)
fig.colorbar(im3, ax=ax3, label=f"RM / ({u.rad / u.m**2:latex_inline})")
ax3.set(title="Peak RM (I-corrected)")

### The distortion the correction removes

Running RM-synthesis on the raw flux-scale Q/U (no Stokes I division) leaves the spectral-index distortion in place. Comparing the two FDFs at the bright pixel, the uncorrected main lobe is a distorted width, set partly by the Stokes I spectrum rather than by the RMSF alone -- spurious Faraday structure. The I-corrected FDF's shape is set only by the RMSF and the true Faraday depth.

In [ ]:
result_raw = rmsynth_3d(
    da.from_array(stokes_q_obs, chunks=chunks),
    da.from_array(stokes_u_obs, chunks=chunks),
    freq_hz,
    phi_max_radm2=150.0,
    d_phi_radm2=1.0,
    weight_type="uniform",
)
fdf_raw = result_raw.fdf_dirty_cube.compute()
phi = result_i.phi_arr_radm2

def _norm(a):
    return np.abs(a) / np.abs(a).max()

plt.figure(figsize=(7, 4))
plt.plot(phi, _norm(fdf_raw[:, jb, ib]), label="uncorrected (Q/U)")
plt.plot(phi, _norm(fdf_i[:, jb, ib]), label="I-corrected (q/u)")
plt.axvline(rm_map[jb, ib], color="k", ls=":", lw=1, label="true RM")
plt.xlim(rm_map[jb, ib] - 60, rm_map[jb, ib] + 60)
plt.xlabel(f"Faraday depth / ({u.rad / u.m**2:latex_inline})")
plt.ylabel("Normalised |FDF|")
plt.title("Stokes I spectral index distorts the FDF main lobe")
plt.legend()

## Serialisation: zarr vs FITS

The dirty FDF and RMSF cubes are complex-valued, which matters for how they get written out:

- **zarr** supports complex128 natively, and `dask.array.to_zarr` writes chunk-by-chunk without ever materialising the full cube in memory -- the write scales with chunk size, not cube size, same as the computation itself (with the same buffer-copy overhead noted above, so budget a few times `target_chunk_mb`, not exactly `target_chunk_mb`).
- **FITS** has no native complex dtype, and `astropy.io.fits` needs a full in-memory array to write. Writing an FDF cube to FITS means computing the whole cube first, then splitting it into real/imaginary (or real/imaginary/polarised-intensity) HDUs -- the same convention classic RM-Tools uses for its `FDF_real_dirty.fits`/`FDF_im_dirty.fits`/`FDF_tot_dirty.fits` outputs.

zarr is the better fit for cube-sized outputs; FITS remains useful for interoperability with tools that expect it.

In [ ]:
from rm_lite.utils.dask_io import write_zarr_group

zarr_store = tmpdir / "rmsynth3d.zarr"
write_zarr_group(
    zarr_store,
    {"fdf_dirty": result.fdf_dirty_cube, "rmsf": result.rmsf_cube},
)

group = zarr.open(zarr_store)
group["fdf_dirty"].shape, group["fdf_dirty"].dtype

In [ ]:
fits.PrimaryHDU(fdf_dirty_cube.real.astype(np.float32)).writeto(
    tmpdir / "FDF_real_dirty.fits", overwrite=True
)
fits.PrimaryHDU(fdf_dirty_cube.imag.astype(np.float32)).writeto(
    tmpdir / "FDF_im_dirty.fits", overwrite=True
)
fits.PrimaryHDU(np.abs(fdf_dirty_cube).astype(np.float32)).writeto(
    tmpdir / "FDF_tot_dirty.fits", overwrite=True
)

roundtrip = fits.getdata(tmpdir / "FDF_tot_dirty.fits")
np.allclose(roundtrip, np.abs(fdf_dirty_cube), atol=1e-5)

See the 3D RM-CLEAN page for deconvolving these dirty FDF/RMSF cubes.